# MLOps Training 2026/2027 — Task 2
## Notebook 5: Feature Engineering
In this notebook, I will build the features selected from the EDA findings and prepare the train, validation, and test sets for modeling.

All transformations will be fitted using the training split only, then applied to validation and test data using the same fitted objects.

Only information that would be available at prediction time will be used.

## 1. Load the Split Datasets
I will load the train, validation, and test artifacts created in Notebook 3.

In [1]:
import pandas as pd
import numpy as np

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

train = pd.read_csv(
    "artifacts/train.csv",
    parse_dates=date_columns
)

validation = pd.read_csv(
    "artifacts/validation.csv",
    parse_dates=date_columns
)

test = pd.read_csv(
    "artifacts/test.csv",
    parse_dates=date_columns
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 30)
Validation: (14471, 30)
Test: (14472, 30)


## 2. Select Prediction-Time Features
Some columns contain information that would only be known after the order has already progressed or been delivered. Those columns must not be used as model inputs because they would cause data leakage.

I will keep only features that are available at or near the time the order is placed.

### 2.1 Prediction Point
The prediction will be made at the time the order is placed.

Because of this, features that become available only after purchase, such as approval, carrier handoff, or final delivery timestamps, will not be used as model inputs.

In [2]:
leakage_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "late_delivery"
]

identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

## 3. Build Prediction-Time Features
I will create features that are available at the time the order is placed and were supported by the EDA findings.

In [3]:
def build_features(df):
    data = df.copy()

    # Time features available at purchase time
    data["purchase_month"] = data["order_purchase_timestamp"].dt.month
    data["purchase_weekday"] = data["order_purchase_timestamp"].dt.dayofweek
    data["purchase_hour"] = data["order_purchase_timestamp"].dt.hour

    # Estimated delivery window
    data["estimated_window_days"] = (
        data["order_estimated_delivery_date"]
        - data["order_purchase_timestamp"]
    ).dt.total_seconds() / (60 * 60 * 24)

    # Geography
    data["same_state"] = (
        data["customer_state"] == data["seller_state"]
    ).astype(int)

    return data

In [4]:
train_features = build_features(train)
validation_features = build_features(validation)
test_features = build_features(test)

print("Train:", train_features.shape)
print("Validation:", validation_features.shape)
print("Test:", test_features.shape)

Train: (67533, 35)
Validation: (14471, 35)
Test: (14472, 35)


## 4. Select the Final Feature Columns
I will keep the features that are available at prediction time and remove identifiers, post-purchase timestamps, raw coordinates, and other columns that are not needed directly by the model.

In [5]:
feature_columns = [
    # Order information
    "item_count",
    "total_price",
    "total_freight_value",
    "unique_products",
    "unique_sellers",
    "payment_count",
    "total_payment_value",
    "max_installments",
    "payment_type_count",

    # Geography
    "customer_state",
    "seller_state",
    "distance_km",
    "same_state",

    # Time
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_window_days"
]

target_column = "late_delivery"

X_train = train_features[feature_columns].copy()
X_validation = validation_features[feature_columns].copy()
X_test = test_features[feature_columns].copy()

y_train = train_features[target_column].copy()
y_validation = validation_features[target_column].copy()
y_test = test_features[target_column].copy()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

X_train: (67533, 17)
X_validation: (14471, 17)
X_test: (14472, 17)

y_train: (67533,)
y_validation: (14471,)
y_test: (14472,)


## 5. Handle Missing Values
Some numerical features contain a small number of missing values, especially `distance_km` and payment-related columns.

I will fit the imputation values using the training data only, then apply the same values to validation and test.

In [6]:
missing_train = (
    X_train.isna().sum()
    .to_frame("missing_count")
)

missing_train["missing_percentage"] = (
    X_train.isna().mean() * 100
).round(2)

missing_train = missing_train[
    missing_train["missing_count"] > 0
]

missing_train

,missing_count,missing_percentage
payment_count,1,0.00
total_payment_value,1,0.00
max_installments,1,0.00
payment_type_count,1,0.00
distance_km,345,0.51


### 5.1 Fit the Numerical Imputer
I will use median imputation for missing numerical values. The imputer will be fitted on the training data only, then applied to validation and test using the same fitted values.

In [7]:
from sklearn.impute import SimpleImputer

numeric_features = [
    "item_count",
    "total_price",
    "total_freight_value",
    "unique_products",
    "unique_sellers",
    "payment_count",
    "total_payment_value",
    "max_installments",
    "payment_type_count",
    "distance_km",
    "same_state",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_window_days"
]

categorical_features = [
    "customer_state",
    "seller_state"
]

numeric_imputer = SimpleImputer(strategy="median")

numeric_imputer.fit(X_train[numeric_features])

X_train[numeric_features] = numeric_imputer.transform(
    X_train[numeric_features]
)

X_validation[numeric_features] = numeric_imputer.transform(
    X_validation[numeric_features]
)

X_test[numeric_features] = numeric_imputer.transform(
    X_test[numeric_features]
)

print("Missing values after numerical imputation:")
print("Train:", X_train[numeric_features].isna().sum().sum())
print("Validation:", X_validation[numeric_features].isna().sum().sum())
print("Test:", X_test[numeric_features].isna().sum().sum())

Missing values after numerical imputation:
Train: 0
Validation: 0
Test: 0


## 6. Encode Categorical Features
The customer and seller state columns are categorical, so they need to be converted into numerical features before modeling.

The encoder will be fitted on the training data only and then applied to validation and test.

In [8]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(X_train[categorical_features])

train_encoded = encoder.transform(
    X_train[categorical_features]
)

validation_encoded = encoder.transform(
    X_validation[categorical_features]
)

test_encoded = encoder.transform(
    X_test[categorical_features]
)

encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

print("Encoded categorical features:", len(encoded_feature_names))
print(encoded_feature_names[:10])

Encoded categorical features: 49
['customer_state_AC' 'customer_state_AL' 'customer_state_AM'
 'customer_state_AP' 'customer_state_BA' 'customer_state_CE'
 'customer_state_DF' 'customer_state_ES' 'customer_state_GO'
 'customer_state_MA']


## 7. Build the Final Feature Tables
I will combine the imputed numerical features with the encoded categorical features to create the final train, validation, and test feature tables.

In [9]:
train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

validation_encoded_df = pd.DataFrame(
    validation_encoded,
    columns=encoded_feature_names,
    index=X_validation.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

X_train_final = pd.concat(
    [
        X_train[numeric_features],
        train_encoded_df
    ],
    axis=1
)

X_validation_final = pd.concat(
    [
        X_validation[numeric_features],
        validation_encoded_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test[numeric_features],
        test_encoded_df
    ],
    axis=1
)

print("Final train shape:", X_train_final.shape)
print("Final validation shape:", X_validation_final.shape)
print("Final test shape:", X_test_final.shape)

Final train shape: (67533, 64)
Final validation shape: (14471, 64)
Final test shape: (14472, 64)


## 8. Scale Numerical Features
The numerical features have very different ranges, and several contain large upper-tail values.

I will use `RobustScaler` for the numerical features because it is less sensitive to extreme values than standard scaling. The scaler will be fitted on the training data only and then applied to validation and test.

In [10]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

scaler.fit(X_train_final[numeric_features])

X_train_final[numeric_features] = scaler.transform(
    X_train_final[numeric_features]
)

X_validation_final[numeric_features] = scaler.transform(
    X_validation_final[numeric_features]
)

X_test_final[numeric_features] = scaler.transform(
    X_test_final[numeric_features]
)

print("Scaling completed.")
print("Train:", X_train_final.shape)
print("Validation:", X_validation_final.shape)
print("Test:", X_test_final.shape)

Scaling completed.
Train: (67533, 64)
Validation: (14471, 64)
Test: (14472, 64)


## 9. Save Feature Engineering Artifacts
I will save the final feature tables, target arrays, fitted preprocessing objects, and feature list so Notebook 6 can load them directly without fitting the transformations again.

In [11]:
import joblib
import json

X_train_final.to_csv("artifacts/X_train.csv", index=False)
X_validation_final.to_csv("artifacts/X_validation.csv", index=False)
X_test_final.to_csv("artifacts/X_test.csv", index=False)

y_train.to_csv("artifacts/y_train.csv", index=False)
y_validation.to_csv("artifacts/y_validation.csv", index=False)
y_test.to_csv("artifacts/y_test.csv", index=False)

joblib.dump(
    numeric_imputer,
    "artifacts/numeric_imputer.joblib"
)

joblib.dump(
    encoder,
    "artifacts/onehot_encoder.joblib"
)

joblib.dump(
    scaler,
    "artifacts/robust_scaler.joblib"
)

with open(
    "artifacts/feature_list.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        list(X_train_final.columns),
        file,
        indent=4
    )

print("Feature engineering artifacts saved successfully!")

Feature engineering artifacts saved successfully!


## 10. Final Check
Before finishing this notebook, I will confirm that the saved feature tables have the same columns, contain no missing values, and can be loaded correctly.

In [12]:
saved_X_train = pd.read_csv("artifacts/X_train.csv")
saved_X_validation = pd.read_csv("artifacts/X_validation.csv")
saved_X_test = pd.read_csv("artifacts/X_test.csv")

print("Shapes:")
print("Train:", saved_X_train.shape)
print("Validation:", saved_X_validation.shape)
print("Test:", saved_X_test.shape)

print("\nMissing values:")
print("Train:", saved_X_train.isna().sum().sum())
print("Validation:", saved_X_validation.isna().sum().sum())
print("Test:", saved_X_test.isna().sum().sum())

print("\nSame feature columns:")
print(
    list(saved_X_train.columns)
    == list(saved_X_validation.columns)
    == list(saved_X_test.columns)
)

Shapes:
Train: (67533, 64)
Validation: (14471, 64)
Test: (14472, 64)

Missing values:
Train: 0
Validation: 0
Test: 0

Same feature columns:
True


## 11. Conclusion
The train, validation, and test datasets were prepared using the same feature engineering process.

Missing numerical values were imputed using values fitted on the training split. Categorical state features were one-hot encoded, and numerical features were scaled using a scaler fitted only on the training data.

The final datasets contain 64 model features and have been saved together with the fitted preprocessing objects and feature list for Notebook 6.